# Quick mechanical-data explorer

This notebook opens the reduced `.txt` file produced by the mechanical reduction script. It reads the column names and units automatically, then opens an interactive plot where you can:

- choose one variable for the **x-axis**;
- show one or several variables on the **y-axis**;
- use the Matplotlib toolbar to **zoom, pan, reset, and save** the figure.

Only the settings in **Cell 2** normally need to be changed. Run the cells from top to bottom.

In [ ]:
# Open the interactive plot in a separate window.
# If you prefer an inline Jupyter widget and have ipympl installed,
# replace 'qt' with 'widget'.
%matplotlib qt

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import RadioButtons, CheckButtons, Button
from IPython.display import display

COLORS = {
    'ink': '#17324D',
    'teal': '#079B8C',
    'panel': '#F2F6F6',
    'grid': '#C8D5DB',
}


In [ ]:
# ------------------------- EDIT THESE SETTINGS -------------------------
DATA_FILE = Path('UCS_granit_acousto_mech.txt')

# Raw ADC columns are hidden by default to keep the selectors short.
INCLUDE_RAW = False

# Preferred variables shown when the explorer first opens.
DEFAULT_X = 'time_s'
DEFAULT_Y = ['axial_stress_MPa']

# Very long files are displayed with at most this many points per curve.
# The original DataFrame remains unchanged.
MAX_DISPLAY_POINTS = 200_000
# -----------------------------------------------------------------------


In [ ]:
def load_reduced_txt(path):
    """Load the two-header-row text file exported by the reduction script.

    Row 1 contains column names and row 2 contains physical units.
    Tab-separated, comma-separated, semicolon-separated, and generic
    whitespace-separated files are accepted.
    """
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            f'File not found: {path.resolve()}\n'
            'Edit DATA_FILE in Cell 2.'
        )

    with path.open('r', encoding='utf-8-sig') as stream:
        name_line = stream.readline().rstrip('\n\r')
        unit_line = stream.readline().rstrip('\n\r')

    # The reduction script normally writes tabs. The alternatives make
    # this small loader useful for later exports as well.
    if '\t' in name_line:
        separator = '\t'
        split_character = '\t'
    elif ',' in name_line:
        separator = ','
        split_character = ','
    elif ';' in name_line:
        separator = ';'
        split_character = ';'
    else:
        separator = r'\s+'
        split_character = None

    names = name_line.split(split_character)
    units_row = unit_line.split(split_character)
    units_row += [''] * max(0, len(names) - len(units_row))
    units = {name.strip(): unit.strip() for name, unit in zip(names, units_row)}

    data = pd.read_csv(
        path, sep=separator, header=0, skiprows=[1], engine='python',
        na_values=['', 'nan', 'NaN', 'None'],
    )
    data.columns = [str(column).strip() for column in data.columns]

    # Convert every exported channel to numeric data. Empty columns are
    # kept in the DataFrame but omitted from the plot selectors.
    for column in data.columns:
        data[column] = pd.to_numeric(data[column], errors='coerce')

    if data.empty:
        raise ValueError('The file contains column names but no data rows.')

    return data, units


mechanical, units = load_reduced_txt(DATA_FILE)
usable_columns = [
    column for column in mechanical.columns
    if np.isfinite(mechanical[column].to_numpy(dtype=float)).any()
]

print(f'Loaded: {DATA_FILE.name}')
print(f'Rows: {len(mechanical):,}')
print(f'Usable numeric columns: {len(usable_columns)}')
display(mechanical[usable_columns].head())


In [ ]:
def open_mechanical_explorer(
    data, units, default_x=None, default_y=None,
    include_raw=False, max_display_points=200_000,
):
    """Open an interactive X-Y explorer using only Matplotlib.

    Radio buttons select one x-variable. Check boxes show or hide any
    number of y-variables. Zoom and pan are provided by the standard
    Matplotlib toolbar.
    """
    candidates = []
    for column in data.columns:
        values = pd.to_numeric(data[column], errors='coerce').to_numpy(dtype=float)
        if np.isfinite(values).any() and (include_raw or not column.endswith('_raw')):
            candidates.append(column)

    if len(candidates) < 2:
        raise ValueError('At least two non-empty numeric columns are required.')

    if default_x not in candidates:
        default_x = 'time_s' if 'time_s' in candidates else candidates[0]

    requested_y = default_y if isinstance(default_y, (list, tuple)) else [default_y]
    selected_y = [column for column in requested_y if column in candidates and column != default_x]
    if not selected_y:
        selected_y = [next(column for column in candidates if column != default_x)]

    # Downsampling only affects display speed, never the loaded dataset.
    stride = max(1, int(np.ceil(len(data) / max_display_points)))
    view = data.iloc[::stride]

    figure = plt.figure(figsize=(14.5, 8.2))
    figure.canvas.manager.set_window_title(f'Mechanical explorer — {DATA_FILE.name}')
    plot_axis = figure.add_axes([0.34, 0.12, 0.63, 0.81])
    x_axis = figure.add_axes([0.025, 0.55, 0.27, 0.38], facecolor=COLORS['panel'])
    y_axis = figure.add_axes([0.025, 0.12, 0.27, 0.38], facecolor=COLORS['panel'])
    reset_axis = figure.add_axes([0.80, 0.025, 0.16, 0.055])

    x_selector = RadioButtons(x_axis, candidates, active=candidates.index(default_x))
    y_active = [column in selected_y for column in candidates]
    y_selector = CheckButtons(y_axis, candidates, actives=y_active)
    reset_button = Button(reset_axis, 'Reset view')

    x_axis.set_title('X variable', fontsize=11, loc='left', color=COLORS['ink'])
    y_axis.set_title('Y variables', fontsize=11, loc='left', color=COLORS['ink'])
    for label in list(x_selector.labels) + list(y_selector.labels):
        label.set_fontsize(8.2)

    state = {'x': default_x, 'visible_y': set(selected_y)}
    palette = plt.colormaps['tab10']
    lines = {}

    def axis_label(column):
        unit = units.get(column, '')
        return f'{column} [{unit}]' if unit and unit != '-' else column

    def redraw(reset_limits=True):
        for line in lines.values():
            line.remove()
        lines.clear()

        x_values = view[state['x']].to_numpy(dtype=float)
        for index, column in enumerate(candidates):
            if column not in state['visible_y'] or column == state['x']:
                continue
            line, = plot_axis.plot(
                x_values, view[column].to_numpy(dtype=float),
                lw=1.2, color=palette(index % 10), label=axis_label(column),
            )
            lines[column] = line

        plot_axis.set_xlabel(axis_label(state['x']))
        if len(lines) == 1:
            plot_axis.set_ylabel(axis_label(next(iter(lines))))
        else:
            plot_axis.set_ylabel('Selected mechanical variables')
        plot_axis.set_title('Mechanical data exploration', color=COLORS['ink'])
        plot_axis.grid(True, color=COLORS['grid'], alpha=0.55)
        if lines:
            plot_axis.legend(loc='best', fontsize=9)
        legend = plot_axis.get_legend()
        if not lines and legend is not None:
            legend.remove()
        if reset_limits:
            plot_axis.relim()
            plot_axis.autoscale_view()
        figure.canvas.draw_idle()

    def choose_x(column):
        state['x'] = column
        redraw(reset_limits=True)

    def toggle_y(column):
        if column in state['visible_y']:
            state['visible_y'].remove(column)
        else:
            state['visible_y'].add(column)
        redraw(reset_limits=True)

    def reset_view(_event):
        plot_axis.relim()
        plot_axis.autoscale_view()
        figure.canvas.draw_idle()

    x_selector.on_clicked(choose_x)
    y_selector.on_clicked(toggle_y)
    reset_button.on_clicked(reset_view)
    redraw()
    plt.show(block=False)

    # Keep the widgets alive for as long as the plot window exists.
    figure._mechanical_explorer_widgets = (x_selector, y_selector, reset_button)
    return figure


In [ ]:
explorer = open_mechanical_explorer(
    mechanical,
    units,
    default_x=DEFAULT_X,
    default_y=DEFAULT_Y,
    include_raw=INCLUDE_RAW,
    max_display_points=MAX_DISPLAY_POINTS,
)
